In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce
import boto3
import datetime

In [0]:
BRONZE_PATH = "s3a://nyc-lakehouse/bronze/"
SILVER_PATH = "s3a://nyc-lakehouse/silver/yellow_taxi"
QUARANTINED_PATH = "s3a://nyc-lakehouse/quarantine/"
bucket_name = "nyc-lakehouse"


In [0]:
files = [f.path for f in dbutils.fs.ls(BRONZE_PATH) if f.path.endswith(".parquet")]

In [0]:

numeric_cols_to_double = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "RatecodeID"
]

int_cols_to_long = [
    "VendorID",
    "payment_type",
    "PULocationID",
    "DOLocationID"]

dfs = []

for file in files:
    df = spark.read.parquet(file)

    for column in numeric_cols_to_double:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast("double"))

    for column in int_cols_to_long:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast("long"))

    dfs.append(df)


In [0]:

bronze_df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)


In [0]:

print(f"Bronze layer record count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze layer record count: 40236152
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [0]:

# ---------------- CONFIG ---------------- #

QUALITY_CONFIG = {
    "max_passenger": 8,
    "max_distance": 100,
    "max_fare": 500,
    "max_total": 1000,
    "max_speed": 100,
    "min_duration": 1,
    "max_duration": 300
}


# ---------------- MAIN FUNCTION ---------------- #

def get_silver_df(bronze_df, config=QUALITY_CONFIG):

    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ---------------- Metadata ---------------- #
    df = bronze_df.withColumn("ingestion_timestamp", lit(ts))

    # ---------------- Dedup (Deterministic) ---------------- #
    df = df.dropDuplicates([
        "VendorID",
        "tpep_pickup_datetime", 
        "tpep_dropoff_datetime", 
        "PULocationID", 
        "DOLocationID"
    ])

    # ---------------- Validation Rules ---------------- #
    df = df.withColumn(
        "invalid_reason",
        when(col("passenger_count") <= 0, "INVALID_PASSENGER")
        .when(col("passenger_count") > config["max_passenger"], "EXCEEDS_MAX_PASSENGER")
        .when(col("trip_distance") <= 0, "INVALID_DISTANCE")
        .when(col("trip_distance") > config["max_distance"], "EXCEEDS_MAX_DISTANCE")
        .when(col("fare_amount") <= 0, "INVALID_FARE")
        .when(col("fare_amount") > config["max_fare"], "EXCEEDS_MAX_FARE")
        .when(col("total_amount") <= 0, "INVALID_TOTAL")
        .when(col("total_amount") > config["max_total"], "EXCEEDS_MAX_TOTAL")
        .when(col("tpep_pickup_datetime").isNull(), "NULL_PICKUP_TIME")
        .when(col("tpep_dropoff_datetime").isNull(), "NULL_DROPOFF_TIME")
        .when(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"), "INVALID_TIME_RANGE")
    )

    df_valid = df.filter(col("invalid_reason").isNull())
    df_quarantine = (
        df.filter(col("invalid_reason").isNotNull())
          .withColumn("quarantine_timestamp", lit(ts))
    )

    # ---------------- Derived Features ---------------- #
    df_valid = (
        df_valid
        .withColumn(
            "trip_duration_minutes",
            (unix_timestamp(col("tpep_dropoff_datetime")) -
             unix_timestamp(col("tpep_pickup_datetime"))) / 60
        )
        .withColumn(
            "average_speed_mph",
            when(col("trip_duration_minutes") > 0,
                 (col("trip_distance") / col("trip_duration_minutes")) * 60)
            .otherwise(lit(0))
        )
        .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))
        .withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
        .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime")))
        .withColumn(
            "is_weekend",
            when(col("pickup_day_of_week").isin([1, 7]), lit(True))
            .otherwise(lit(False))
        )
        .withColumn(
            "tip_percentage",
            when(col("fare_amount") > 0,
                 (col("tip_amount") / col("fare_amount")) * 100)
            .otherwise(lit(0))
        )
    )

    # ---------------- Data Quality Flag ---------------- #
    df_valid = df_valid.withColumn(
        "data_quality_flag",
        when(
            (col("average_speed_mph") > 0) &
            (col("average_speed_mph") < config["max_speed"]) &
            (col("trip_duration_minutes") > config["min_duration"]) &
            (col("trip_duration_minutes") < config["max_duration"]),
            lit("VALID")
        ).otherwise(lit("REVIEW"))
    )

    # ---------------- Processing Metadata ---------------- #
    df_valid = df_valid.withColumn(
        "silver_processing_timestamp",
        lit(ts)
    )

    # ---------------- Audit Metrics ---------------- #
    audit_metrics = {
        "total_records": bronze_df.count(),
        "valid_records": df_valid.count(),
        "invalid_records": df_quarantine.count()
    }

    print("---- AUDIT METRICS ----")
    for k, v in audit_metrics.items():
        print(f"{k}: {v}")

    return df_valid, df_quarantine


In [0]:
silver_df, df_quarantined = get_silver_df(bronze_df)
#silver_df.printSchema()

---- AUDIT METRICS ----
total_records: 40236152
valid_records: 35678614
invalid_records: 3923355



### One time code to create delta table

silver_df.createOrReplaceTempView("tmp_silver")

spark.sql("""
CREATE TABLE nyc_silver_yellow
USING DELTA
PARTITIONED BY (pickup_date)
LOCATION 's3a://nyc-lakehouse/silver/yellow_taxi'
AS
SELECT * FROM tmp_silver
""")



### To drop the delta table
spark.sql("""DROP TABLE nyc_silver_yellow""")

In [0]:
def merge_into_silver(spark, df_valid, silver_path):

    if not DeltaTable.isDeltaTable(spark, silver_path):

        (
            df_valid.write
            .format("delta")
            .partitionBy("pickup_date")
            .mode("overwrite")
            .save(silver_path)
        )

        print("Silver table created.")
        return

    delta_table = DeltaTable.forPath(spark, silver_path)

    merge_condition = """
        target.tpep_pickup_datetime = source.tpep_pickup_datetime
        AND target.tpep_dropoff_datetime = source.tpep_dropoff_datetime
        AND target.PULocationID = source.PULocationID
        AND target.DOLocationID = source.DOLocationID
        AND target.vendorID = source.vendorID
    """

    (
        delta_table.alias("target")
        .merge(
            df_valid.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver table MERGE completed.")

In [0]:
def write_quarantine(df_quarantine, quarantine_path):


    (
        df_quarantine.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(quarantine_path)
    )

    print("Quarantine records written.")

In [0]:
merge_into_silver(spark, silver_df, SILVER_PATH)

Silver table MERGE completed.


In [0]:
write_quarantine(df_quarantined, QUARANTINED_PATH)


Quarantine records written.


In [0]:
#------------------------------Move files to processed folder-----------------------------#
# Initialize client
ACCESS_KEY = dbutils.secrets.get(scope="my_app_creds", key="access_key_id")
SECRET_KEY = dbutils.secrets.get(scope="my_app_creds", key="secret_access_key")
REGION = 'us-east-1'

# Create a client with explicit credentials
s3_client = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION
)


source_prefix = "bronze/"     # e.g. data/input/
destination_prefix = "bronze/processed/" # e.g. data/archive/

# List objects in source folder
response = s3_client.list_objects_v2(
    Bucket=bucket_name,
    Prefix=source_prefix
)


for file in files:
        source_key = file.replace("s3a://"+bucket_name+"/", "", 1)
        
        # Skip folder itself
        if source_key.endswith("/"):
            continue
        
        # Build destination key
        dest_key = source_key.replace(source_prefix, destination_prefix, 1)
        
        print(f"Moving {source_key} --> {dest_key}")
        
        # Copy
        s3_client.copy_object(
            Bucket=bucket_name,
            CopySource={
                "Bucket": bucket_name,
                "Key": source_key
            },
            Key=dest_key
        )
        
        # Delete original
        s3_client.delete_object(
            Bucket=bucket_name,
            Key=source_key
        )
        
print("Move completed.")



Moving bronze/yellow_tripdata_2025-01.parquet --> bronze/processed/yellow_tripdata_2025-01.parquet
Moving bronze/yellow_tripdata_2025-02.parquet --> bronze/processed/yellow_tripdata_2025-02.parquet
Moving bronze/yellow_tripdata_2025-03.parquet --> bronze/processed/yellow_tripdata_2025-03.parquet
Moving bronze/yellow_tripdata_2025-04.parquet --> bronze/processed/yellow_tripdata_2025-04.parquet
Moving bronze/yellow_tripdata_2025-05.parquet --> bronze/processed/yellow_tripdata_2025-05.parquet
Moving bronze/yellow_tripdata_2025-06.parquet --> bronze/processed/yellow_tripdata_2025-06.parquet
Moving bronze/yellow_tripdata_2025-07.parquet --> bronze/processed/yellow_tripdata_2025-07.parquet
Moving bronze/yellow_tripdata_2025-08.parquet --> bronze/processed/yellow_tripdata_2025-08.parquet
Moving bronze/yellow_tripdata_2025-09.parquet --> bronze/processed/yellow_tripdata_2025-09.parquet
Moving bronze/yellow_tripdata_2025-10.parquet --> bronze/processed/yellow_tripdata_2025-10.parquet
Move compl

# For analysis purpose
w = Window.partitionBy("VendorID", "tpep_dropoff_datetime", "tpep_pickup_datetime" , "PULocationID")
temp_df = bronze_df.withColumn('count', count('*').over(w)).filter(col('count') > 1)
temp_df.show(5)


temp = spark.read.format('delta').table('nyc_silver_yellow')
temp.count()
